# Step 3: Build Sliding Windows

This notebook converts cycle-level battery data into sequence-model-ready sequences.

The goal is to create `X_train`, `y_train`, `X_test`, and `y_test` without mixing batteries or using `cycle` as a model input.

In [ ]:
# Import the tools and locate the cycle-level files so the notebook can run from either the project root or the notebook folder.
from pathlib import Path

import numpy as np
import pandas as pd

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "processed" / "cycle_train_dataset.csv").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

CYCLE_TRAIN_FILE = PROJECT_ROOT / "data" / "processed" / "cycle_train_dataset.csv"
CYCLE_VALIDATION_FILE = PROJECT_ROOT / "data" / "processed" / "cycle_validation_dataset.csv"
CYCLE_TEST_FILE = PROJECT_ROOT / "data" / "processed" / "cycle_test_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Cycle train file:", CYCLE_TRAIN_FILE)
print("Cycle validation file:", CYCLE_VALIDATION_FILE)
print("Cycle test file:", CYCLE_TEST_FILE)

In [ ]:
# Load the cycle-level train, validation, and test datasets because each row is now one battery cycle.
train_cycle_df = pd.read_csv(CYCLE_TRAIN_FILE)
validation_cycle_df = pd.read_csv(CYCLE_VALIDATION_FILE)
test_cycle_df = pd.read_csv(CYCLE_TEST_FILE)

print("Train cycle shape:", train_cycle_df.shape)
print("Validation cycle shape:", validation_cycle_df.shape)
print("Test cycle shape:", test_cycle_df.shape)
print("Train batteries:", sorted(train_cycle_df["battery_id"].unique()))
print("Validation batteries:", sorted(validation_cycle_df["battery_id"].unique()))
print("Test batteries:", sorted(test_cycle_df["battery_id"].unique()))

train_cycle_df.head()

In [ ]:
# Define the sequence settings once so the feature list, target, and window size stay consistent everywhere.
FEATURE_COLUMNS = ["chI", "chV", "chT", "disI", "disV", "BCt", "SOH"]
TARGET_COLUMN = "RUL"
ID_COLUMN = "battery_id"
ORDER_COLUMN = "cycle"
WINDOW_SIZE = 10

required_columns = [ORDER_COLUMN, *FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]

for name, df in [("train", train_cycle_df), ("validation", validation_cycle_df), ("test", test_cycle_df)]:
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{name} data is missing required columns: {missing_columns}")

print("Window size:", WINDOW_SIZE)
print("Input features:", FEATURE_COLUMNS)
print("Target:", TARGET_COLUMN)
print("Used only for sorting/grouping:", [ID_COLUMN, ORDER_COLUMN])

In [ ]:
# Inspect cycle counts before making windows so we know how many sequences each battery can produce.
train_cycle_counts = train_cycle_df.groupby(ID_COLUMN)[ORDER_COLUMN].count()
validation_cycle_counts = validation_cycle_df.groupby(ID_COLUMN)[ORDER_COLUMN].count()
test_cycle_counts = test_cycle_df.groupby(ID_COLUMN)[ORDER_COLUMN].count()

train_expected_windows = (train_cycle_counts - WINDOW_SIZE + 1).clip(lower=0)
validation_expected_windows = (validation_cycle_counts - WINDOW_SIZE + 1).clip(lower=0)
test_expected_windows = (test_cycle_counts - WINDOW_SIZE + 1).clip(lower=0)

print("Expected train windows per battery:")
print(train_expected_windows)
print("Total expected train windows:", int(train_expected_windows.sum()))

print("\nExpected validation windows per battery:")
print(validation_expected_windows)
print("Total expected validation windows:", int(validation_expected_windows.sum()))

print("\nExpected test windows per battery:")
print(test_expected_windows)
print("Total expected test windows:", int(test_expected_windows.sum()))

In [ ]:
# Build sliding windows inside each battery only so a sequence never jumps from one battery to another.
def create_sliding_windows(df, window_size, feature_columns, target_column, id_column, order_column):
    X_windows = []
    y_values = []
    metadata = []

    for battery_id, battery_df in df.groupby(id_column):
        battery_df = battery_df.sort_values(order_column).reset_index(drop=True)

        feature_values = battery_df[feature_columns].to_numpy(dtype=np.float32)
        target_values = battery_df[target_column].to_numpy(dtype=np.float32)
        cycle_values = battery_df[order_column].to_numpy()

        max_start = len(battery_df) - window_size + 1
        if max_start <= 0:
            continue

        for start_idx in range(max_start):
            end_idx = start_idx + window_size
            target_idx = end_idx - 1

            X_windows.append(feature_values[start_idx:end_idx])
            y_values.append(target_values[target_idx])
            metadata.append(
                {
                    "battery_id": battery_id,
                    "start_cycle": cycle_values[start_idx],
                    "end_cycle": cycle_values[target_idx],
                    "target_RUL": target_values[target_idx],
                }
            )

    X = np.array(X_windows, dtype=np.float32)
    y = np.array(y_values, dtype=np.float32)
    window_metadata = pd.DataFrame(metadata)

    return X, y, window_metadata

In [ ]:
# Create the train, validation, and test arrays; X is 3D and y is one RUL value per window.
X_train, y_train, train_window_metadata = create_sliding_windows(
    train_cycle_df,
    WINDOW_SIZE,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    ID_COLUMN,
    ORDER_COLUMN,
)

X_val, y_val, validation_window_metadata = create_sliding_windows(
    validation_cycle_df,
    WINDOW_SIZE,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    ID_COLUMN,
    ORDER_COLUMN,
)

X_test, y_test, test_window_metadata = create_sliding_windows(
    test_cycle_df,
    WINDOW_SIZE,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    ID_COLUMN,
    ORDER_COLUMN,
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Validate the shapes because shape mistakes are one of the easiest ways to break a sequence-model pipeline.
expected_train_windows = int(train_expected_windows.sum())
expected_validation_windows = int(validation_expected_windows.sum())
expected_test_windows = int(test_expected_windows.sum())
expected_feature_count = len(FEATURE_COLUMNS)

assert X_train.shape == (expected_train_windows, WINDOW_SIZE, expected_feature_count)
assert y_train.shape == (expected_train_windows,)
assert X_val.shape == (expected_validation_windows, WINDOW_SIZE, expected_feature_count)
assert y_val.shape == (expected_validation_windows,)
assert X_test.shape == (expected_test_windows, WINDOW_SIZE, expected_feature_count)
assert y_test.shape == (expected_test_windows,)

print("Shape validation passed.")

In [ ]:
# Inspect the first window to connect the array shape back to real battery cycles.
first_window = pd.DataFrame(X_train[0], columns=FEATURE_COLUMNS)
first_window_info = train_window_metadata.iloc[0]

print("First window battery:", first_window_info["battery_id"])
print("Start cycle:", first_window_info["start_cycle"])
print("End cycle:", first_window_info["end_cycle"])
print("Target RUL:", first_window_info["target_RUL"])

first_window

In [ ]:
# Save the window arrays and metadata so model notebooks can load exactly the same sequences.
WINDOW_OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / f"windows_w{WINDOW_SIZE}.npz"
TRAIN_METADATA_FILE = PROJECT_ROOT / "data" / "processed" / f"train_window_metadata_w{WINDOW_SIZE}.csv"
VALIDATION_METADATA_FILE = PROJECT_ROOT / "data" / "processed" / f"validation_window_metadata_w{WINDOW_SIZE}.csv"
TEST_METADATA_FILE = PROJECT_ROOT / "data" / "processed" / f"test_window_metadata_w{WINDOW_SIZE}.csv"

np.savez(
    WINDOW_OUTPUT_FILE,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    feature_columns=np.array(FEATURE_COLUMNS),
    window_size=np.array(WINDOW_SIZE),
)

train_window_metadata.to_csv(TRAIN_METADATA_FILE, index=False)
validation_window_metadata.to_csv(VALIDATION_METADATA_FILE, index=False)
test_window_metadata.to_csv(TEST_METADATA_FILE, index=False)

print("Saved windows to:", WINDOW_OUTPUT_FILE)
print("Saved train metadata to:", TRAIN_METADATA_FILE)
print("Saved validation metadata to:", VALIDATION_METADATA_FILE)
print("Saved test metadata to:", TEST_METADATA_FILE)